In [ ]:
# VEGF Docking Preparation Notebook
# Windows + Conda + VSCode compatible

# Step 0: Install needed packages (uncomment if running first time)
# !conda install -c conda-forge biopython openbabel meeko -y
# !pip install requests

In [18]:

from Bio.PDB import PDBList, PDBParser, Select, PDBIO
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy
import subprocess
import tempfile
from openbabel import pybel


In [19]:
# --- User parameters ---
pdb_id = "1VPF"                  # VEGF-A dimer
output_dir = "DockingPrep"
peptide_csv = "../data/processed/sequences_reviewed.csv"     # Your CSV file with sequences

os.makedirs(output_dir, exist_ok=True)
ligand_dir = os.path.join(output_dir, "ligands")
os.makedirs(ligand_dir, exist_ok=True)

In [4]:
# --- Step 1: Download VEGF structure ---
print(f"Downloading PDB {pdb_id}...")
pdbl = PDBList()
pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir=output_dir, file_format="pdb")
print(f"Downloaded: {pdb_file}")

Downloaded: DockingPrep\pdb1vpf.ent


In [5]:
# --- Step 2: Clean protein (remove waters & heteroatoms) ---
class ProteinSelect(Select):
    def accept_residue(self, residue):
        # Keep only standard amino acids
        return residue.id[0] == " "

parser = PDBParser(QUIET=True)
structure = parser.get_structure(pdb_id, pdb_file)

clean_pdb_path = os.path.join(output_dir, f"{pdb_id}_clean.pdb")
io = PDBIO()
io.set_structure(structure)
io.save(clean_pdb_path, ProteinSelect())
print(f"Clean PDB saved: {clean_pdb_path}")

Clean PDB saved: DockingPrep\1VPF_clean.pdb


In [6]:
# --- Step 3: Prepare receptor with OpenBabel ----
# Add polar hydrogens and convert to PDBQT
pdbqt_path = os.path.join(output_dir, f"{pdb_id}_receptor.pdbqt")

# Run OpenBabel command
cmd = f'obabel "{clean_pdb_path}" -O "{pdbqt_path}" -xr -xh -p'
# -xr = remove water
# -xh = add hydrogens
# -p  = optimize hydrogen positions
os.system(cmd)

print("\n✅ VEGF receptor ready for docking")
print(f"Cleaned protein: {clean_pdb_path}")
print(f"Receptor PDBQT: {pdbqt_path}")


✅ VEGF receptor ready for docking
Cleaned protein: DockingPrep\1VPF_clean.pdb
Receptor PDBQT: DockingPrep\1VPF_receptor.pdbqt


### Peptide preparation

In [20]:
# --- Step 4: Load peptides from CSV ---
df = pd.read_csv(peptide_csv)
print(f"Loaded {len(df)} peptides from {peptide_csv}")

Loaded 6 peptides from ../data/processed/sequences_reviewed.csv


In [21]:
# Convert each peptide sequence to MOL2 using Open Babel
for i, row in df.iterrows():
    seq = row["Sequence"]
    ligand_name = f"peptide_{i+1}"

    fasta_path = os.path.join(ligand_dir, f"{ligand_name}.fasta")
    mol2_path = os.path.join(ligand_dir, f"{ligand_name}.mol2")

    # Write FASTA
    with open(fasta_path, "w") as f:
        f.write(f">ligand_{i+1}\n{seq}\n")

    # Convert FASTA → MOL2 with 3D coordinates using Open Babel
    subprocess.run([
        "obabel",
        fasta_path,
        "-O", mol2_path,
        "--gen3d",
        "--addtotitle", ""
    ], check=True)

    print(f"✅ MOL2 created for {seq}: {mol2_path}")

✅ MOL2 created for VAGKVAIRKDVPEISGG: DockingPrep\ligands\peptide_1.mol2
✅ MOL2 created for WKTMSPKIVLEDKQALF: DockingPrep\ligands\peptide_2.mol2
✅ MOL2 created for WWCLECWCYIVKKKQNG: DockingPrep\ligands\peptide_3.mol2
✅ MOL2 created for KKWPKYCFCLVIEVLGS: DockingPrep\ligands\peptide_4.mol2
✅ MOL2 created for PIGICVYPSRNFVPSKD: DockingPrep\ligands\peptide_5.mol2
✅ MOL2 created for GHNPPQIKYLAFANHLY: DockingPrep\ligands\peptide_6.mol2
